In [2]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# 1. Take User Input (Combined lines)
rows, cols = map(int, input("Enter grid rows and cols (space-separated): ").split())
target, temp = map(float, input("Enter target moisture % and temp °C (space-separated): ").split())

# Initialize starting soil moisture and water schedule
np.random.seed(42)
moisture = np.random.randint(20, 50, size=(rows, cols)).astype(float)
water_schedule = np.zeros((rows, cols))
evaporation = temp * 0.1

def update_cell(pos):
    r, c = pos
    neighbors = [
        moisture[r + dr, c + dc]
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]
        if 0 <= r + dr < rows and 0 <= c + dc < cols
    ]
    neighbor_avg = np.mean(neighbors) if neighbors else moisture[r, c]

    seepage = (neighbor_avg - moisture[r, c]) * 0.1
    water_needed = max(0.0, target - (moisture[r, c] + seepage))
    new_moisture = moisture[r, c] + water_needed + seepage - evaporation

    return r, c, water_needed, new_moisture

# 2. Run Parallel Cellular Algorithm (3 time steps)
cells = [(r, c) for r in range(rows) for c in range(cols)]

for _ in range(3):
    next_moisture = np.copy(moisture)

    with ThreadPoolExecutor() as executor:
        results = executor.map(update_cell, cells)

    for r, c, water, new_m in results:
        water_schedule[r, c] += water
        next_moisture[r, c] = new_m
    moisture = next_moisture

# 3. Short Output
print("\nFinal Soil Moisture (%):")
print(np.round(moisture, 1))

print("\nWater Scheduled (Liters):")
print(np.round(water_schedule, 1))
print(f"\nTotal Water Used: {np.sum(water_schedule):.1f} L")


Enter grid rows and cols (space-separated): 3 3
Enter target moisture % and temp °C (space-separated): 70 30

Final Soil Moisture (%):
[[67. 67. 67.]
 [67. 67. 67.]
 [67. 67. 67.]]

Water Scheduled (Liters):
[[49.  37.4 29.5]
 [41.9 45.5 48.2]
 [29.1 36.5 49.2]]

Total Water Used: 366.4 L
